In [9]:
# Importing the dependent moduels
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (train_test_split,
                                    cross_val_score,
                                    GridSearchCV,RandomizedSearchCV
)
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

In [3]:
digits = load_digits()

In [4]:
dir(digits)

['DESCR', 'data', 'feature_names', 'frame', 'images', 'target', 'target_names']

In [6]:
digits.target

array([0, 1, 2, ..., 8, 9, 8])

In [7]:
df = pd.DataFrame(digits.data, columns=digits.feature_names)
df["target"] = digits.target
df.head()

,pixel_0_0,pixel_0_1,pixel_0_2,pixel_0_3,pixel_0_4,pixel_0_5,pixel_0_6,pixel_0_7,pixel_1_0,pixel_1_1,...,pixel_6_7,pixel_7_0,pixel_7_1,pixel_7_2,pixel_7_3,pixel_7_4,pixel_7_5,pixel_7_6,pixel_7_7,target
0,0.0,0.0,5.0,13.0,9.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,6.0,13.0,10.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,12.0,13.0,5.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,11.0,16.0,10.0,0.0,0.0,1
2,0.0,0.0,0.0,4.0,15.0,12.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,3.0,11.0,16.0,9.0,0.0,2
3,0.0,0.0,7.0,15.0,13.0,1.0,0.0,0.0,0.0,8.0,...,0.0,0.0,0.0,7.0,13.0,13.0,9.0,0.0,0.0,3
4,0.0,0.0,0.0,1.0,11.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,2.0,16.0,4.0,0.0,0.0,4


In [12]:
X_train, X_test, Y_train, Y_test = train_test_split(digits.data, digits.target, stratify=digits.target, test_size=0.2)

In [13]:
df.shape, X_train.shape, X_test.shape

((1797, 65), (1437, 64), (360, 64))

In [21]:
model = SVC(kernel="linear", C=10, gamma="auto")
model.fit(X_train, Y_train)
model.score(X_test, Y_test)

0.9861111111111112

In [22]:
cross_val_score(SVC(kernel="rbf", C=10, gamma="auto"), digits.data, digits.target, cv=5)

array([0.45277778, 0.46944444, 0.47910864, 0.47910864, 0.50139276])

In [23]:
cross_val_score(SVC(kernel="rbf", C=20, gamma="auto"), digits.data, digits.target, cv=5)

array([0.45277778, 0.46944444, 0.47910864, 0.47910864, 0.50139276])

In [24]:
cross_val_score(SVC(kernel="linear", C=10, gamma="auto"), digits.data, digits.target, cv=5)

array([0.96388889, 0.91944444, 0.96657382, 0.9637883 , 0.92479109])

In [26]:
kernels = ["rbf", "linear"]
C = [1,10,20]
avg_scores = {}
for kval in kernels:
    for cval in C:
        cv_scores = cross_val_score(SVC(kernel=kval, C=cval, gamma="auto"), digits.data, digits.target, cv=5)
        avg_scores[kval + "_" + str(cval)] = np.average(cv_scores)

avg_scores

{'rbf_1': 0.448545341999381,
 'rbf_10': 0.47636645001547506,
 'rbf_20': 0.47636645001547506,
 'linear_1': 0.9476973073351903,
 'linear_10': 0.9476973073351903,
 'linear_20': 0.9476973073351903}

In [27]:
# do the same thing wuth sklearn grid seach cv
clf = GridSearchCV(SVC(gamma="auto"),  {
    "C": [1,10,20],
    "kernel": ["rbf", "linear"]
}, cv=5, return_train_score=False)
clf.fit(digits.data, digits.target)
clf.cv_results_

{'mean_fit_time': array([0.2920918 , 0.03404975, 0.27982945, 0.02948794, 0.28660202,
        0.02927918]),
 'std_fit_time': array([0.01110298, 0.00411496, 0.00365774, 0.0010307 , 0.01565584,
        0.00046254]),
 'mean_score_time': array([0.05279865, 0.00885792, 0.05195932, 0.00759611, 0.05526867,
        0.00763688]),
 'std_score_time': array([0.00144863, 0.00092893, 0.00086271, 0.00018991, 0.00568546,
        0.00016156]),
 'param_C': masked_array(data=[1, 1, 10, 10, 20, 20],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'param_kernel': masked_array(data=['rbf', 'linear', 'rbf', 'linear', 'rbf', 'linear'],
              mask=[False, False, False, False, False, False],
        fill_value='?',
             dtype=object),
 'params': [{'C': 1, 'kernel': 'rbf'},
  {'C': 1, 'kernel': 'linear'},
  {'C': 10, 'kernel': 'rbf'},
  {'C': 10, 'kernel': 'linear'},
  {'C': 20, 'kernel': 'rbf'},
  {'C': 20, 'kernel': 'linear'}],


In [29]:
df = pd.DataFrame(clf.cv_results_)
df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.292092,0.011103,0.052799,0.001449,1,rbf,"{'C': 1, 'kernel': 'rbf'}",0.411111,0.450000,0.454039,0.448468,0.479109,0.448545,0.021761,6
1,0.034050,0.004115,0.008858,0.000929,1,linear,"{'C': 1, 'kernel': 'linear'}",0.963889,0.919444,0.966574,0.963788,0.924791,0.947697,0.020978,1
2,0.279829,0.003658,0.051959,0.000863,10,rbf,"{'C': 10, 'kernel': 'rbf'}",0.452778,0.469444,0.479109,0.479109,0.501393,0.476366,0.015784,4
3,0.029488,0.001031,0.007596,0.000190,10,linear,"{'C': 10, 'kernel': 'linear'}",0.963889,0.919444,0.966574,0.963788,0.924791,0.947697,0.020978,1
4,0.286602,0.015656,0.055269,0.005685,20,rbf,"{'C': 20, 'kernel': 'rbf'}",0.452778,0.469444,0.479109,0.479109,0.501393,0.476366,0.015784,4
5,0.029279,0.000463,0.007637,0.000162,20,linear,"{'C': 20, 'kernel': 'linear'}",0.963889,0.919444,0.966574,0.963788,0.924791,0.947697,0.020978,1


In [31]:
df[["param_C", "param_kernel", "mean_test_score"]]

,param_C,param_kernel,mean_test_score
0,1,rbf,0.448545
1,1,linear,0.947697
2,10,rbf,0.476366
3,10,linear,0.947697
4,20,rbf,0.476366
5,20,linear,0.947697


In [32]:
dir(clf)

['__abstractmethods__',
 '__annotations__',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__getstate__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__setstate__',
 '__sizeof__',
 '__sklearn_clone__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_abc_impl',
 '_build_request_for_signature',
 '_check_feature_names',
 '_check_n_features',
 '_check_refit_for_multimetric',
 '_doc_link_module',
 '_doc_link_template',
 '_doc_link_url_param_generator',
 '_estimator_type',
 '_format_results',
 '_get_default_requests',
 '_get_doc_link',
 '_get_metadata_request',
 '_get_param_names',
 '_get_routed_params_for_fit',
 '_get_scorers',
 '_get_tags',
 '_more_tags',
 '_parameter_constraints',
 '_repr_html_',
 '_repr_html_inner',
 '_repr_mimebundle_',
 '_required_parameters',
 '_run

In [33]:
clf.best_score_

0.9476973073351903

In [34]:
clf.best_params_

{'C': 1, 'kernel': 'linear'}

In [37]:
# Use Randomized Search CV to run test in lesser time
rs = RandomizedSearchCV(SVC(gamma="auto"), {
    "C": [1,10,20],
    "kernel": ["rbf", "linear"]
    },
    cv=5,
    return_train_score=False, 
    n_iter=2
)
rs.fit(digits.data, digits.target)
df = pd.DataFrame(rs.cv_results_)[["param_C", "param_kernel", "mean_test_score"]]
df

,param_C,param_kernel,mean_test_score
0,10,linear,0.947697
1,10,rbf,0.476366


In [39]:
# dictionay for model and their parameters
model_params = {
    'svm': {
        'model': SVC(gamma='auto'),
        'params' : {
            'C': [1,10,20],
            'kernel': ['rbf','linear']
        }  
    },
    'random_forest': {
        'model': RandomForestClassifier(),
        'params' : {
            'n_estimators': [1,5,10]
        }
    },
    'logistic_regression' : {
        'model': LogisticRegression(solver='liblinear',multi_class='auto'),
        'params': {
            'C': [1,5,10]
        }
    },
    "decission_tree": {
        "model": DecisionTreeClassifier(),
    },
    "naive_bayes_gs": {
        "model": GaussianNB()
    },
    "naive_bayes_m": {
        "model": MultinomialNB()
    }
}

In [43]:
scores = []

for model_name, mp in model_params.items():
    clf = GridSearchCV(mp["model"], mp.get("params", {}), cv=5, return_train_score=False)
    clf.fit(digits.data, digits.target)
    scores.append({
        "model": model_name,
        "best_score": clf.best_score_,
        "best_param": clf.best_params_
    })
scores

[{'model': 'svm',
  'best_score': 0.9476973073351903,
  'best_param': {'C': 1, 'kernel': 'linear'}},
 {'model': 'random_forest',
  'best_score': 0.9098808418446301,
  'best_param': {'n_estimators': 10}},
 {'model': 'logistic_regression',
  'best_score': 0.9221138966264315,
  'best_param': {'C': 1}},
 {'model': 'decission_tree',
  'best_score': 0.780792324357784,
  'best_param': {}},
 {'model': 'naive_bayes_gs',
  'best_score': 0.8069281956050759,
  'best_param': {}},
 {'model': 'naive_bayes_m',
  'best_score': 0.8703497369235531,
  'best_param': {}}]

In [44]:
df = pd.DataFrame(scores, columns=["model", "best_score", "best_param"])
df

,model,best_score,best_param
0,svm,0.947697,"{'C': 1, 'kernel': 'linear'}"
1,random_forest,0.909881,{'n_estimators': 10}
2,logistic_regression,0.922114,{'C': 1}
3,decission_tree,0.780792,{}
4,naive_bayes_gs,0.806928,{}
5,naive_bayes_m,0.870350,{}
